# 01 Quickstart: llamatelemetry v0.1.0

Get up and running with GGUF inference on **Kaggle (dual T4)** in under 5 minutes.

**What you will learn:**
- Install the SDK from GitHub
- Detect available CUDA GPUs
- Load a small GGUF model via `InferenceEngine`
- Run inference and inspect results
- Use the OpenAI-compatible `LlamaCppClient`

**Requirements:** Kaggle notebook with GPU T4 x2 accelerator enabled.

## 1) Install llamatelemetry

In [ ]:
!pip -q install git+https://github.com/llamatelemetry/llamatelemetry.git@v0.1.0

## 2) Verify GPU availability

In [ ]:
import llamatelemetry as lt
from llamatelemetry import detect_cuda

cuda_info = detect_cuda()
print(f"CUDA available: {cuda_info['available']}")
print(f"CUDA version:   {cuda_info.get('version')}")
for i, gpu in enumerate(cuda_info.get('gpus', [])):
    print(f"  GPU {i}: {gpu}")

## 3) Load a small GGUF model

`InferenceEngine` is the high-level API. It manages the llama-server binary,
downloads models from the built-in registry, and provides `generate()` / `infer()`.

In [ ]:
engine = lt.InferenceEngine(enable_telemetry=False)

# Load a small model from the built-in registry.
# auto_start=True launches llama-server automatically.
engine.load_model("gemma-3-1b-Q4_K_M", auto_start=True)

print(f"Model loaded: {engine.is_loaded}")
print(f"Server healthy: {engine.check_server()}")

## 4) Run inference

`generate()` (alias for `infer()`) returns an `InferResult` with:
- `success` — whether the request completed
- `text` — generated text
- `tokens_generated` — output token count
- `latency_ms` — end-to-end latency
- `tokens_per_sec` — throughput

In [ ]:
result = engine.generate("What is CUDA?", max_tokens=64)

print(f"Success:    {result.success}")
print(f"Tokens:     {result.tokens_generated}")
print(f"Latency:    {result.latency_ms:.1f} ms")
print(f"Throughput: {result.tokens_per_sec:.1f} tok/s")
print(f"\nOutput:\n{result.text}")

## 5) Batch inference

In [ ]:
prompts = [
    "Explain tensor cores in one sentence.",
    "What is llama.cpp?",
    "Define GGUF format briefly.",
]
results = engine.batch_infer(prompts, max_tokens=48)

for prompt, res in zip(prompts, results):
    print(f"Q: {prompt}")
    print(f"A: {res.text[:120]}...\n")

## 6) Engine metrics

In [ ]:
import json

metrics = engine.get_metrics()
print(json.dumps(metrics, indent=2, default=str))

## 7) Alternative: OpenAI-compatible LlamaCppClient

If you prefer the OpenAI chat-completions style API, use `LlamaCppClient` directly.
Note: `ServerManager` defaults to port **8090**, so pass the matching URL.

In [ ]:
from llamatelemetry.api import LlamaCppClient

client = LlamaCppClient(base_url="http://127.0.0.1:8090")

# Convenience method (singular: chat_completion)
resp = client.chat_completion(
    messages=[{"role": "user", "content": "Hello from llama.cpp!"}],
    model="local-gguf",
    max_tokens=32,
)
print(resp.choices[0].message.content)

In [ ]:
# OpenAI-style chained call
resp2 = client.chat.completions.create(
    messages=[{"role": "user", "content": "What is OpenTelemetry?"}],
    model="local-gguf",
    max_tokens=48,
)
print(resp2.choices[0].message.content)

## 8) Cleanup

In [ ]:
engine.unload_model()
print("Done.")